# Challenge 1: House Prices Prediction (Ames Housing) — phiên bản làm lại
**Nhóm 5 – Lớp DCT124C1 – Trường Đại học Sài Gòn** | GVHD: TS. Đỗ Như Tài  
Thành viên: Phạm Minh Khôi (3124411135), Trương Thuận Phú (3124411234), Nguyễn Mạnh Thắng (3124411283), Nguyễn Hoàng Quân (3124411250)

Notebook này chạy **một lượt từ đầu đến cuối** và sinh ra toàn bộ số liệu, hình (`figs/`), bảng (`outputs/`) và `submission.csv` dùng trong paper, slide, poster.

Đặc điểm của quy trình:
1. Chuẩn hóa, chọn đặc trưng (SelectKBest, RF importance) được đặt **bên trong `Pipeline`** nên mỗi fold chỉ học từ tập train của fold đó (không rò rỉ dữ liệu).
2. Tiền xử lý khớp đúng mô tả trong paper (LotFrontage theo Neighborhood, các cột diện tích thiếu → 0).
3. Có **ablation** cho log-transform và loại outlier để kiểm chứng định lượng thay vì chỉ khẳng định.
4. Thêm bước Feature Engineering + xử lý độ lệch + tinh chỉnh siêu tham số + blending, có CV riêng cho mô hình nộp cuối.
5. Mọi hình đều sinh từ code ở đây; hình Actual vs Predicted dùng dự đoán **out-of-fold** (không phải dự đoán trên tập train).

In [1]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
SEED = 42
for d in ('figs', 'outputs'): os.makedirs(d, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
NAVY, BLUE, ORANGE, GREEN, GREY = '#1F3A5F', '#4C78A8', '#F58518', '#54A24B', '#8C8C8C'
RES = {}   # gom số liệu cho paper/slide/poster
print('Môi trường sẵn sàng.')

Môi trường sẵn sàng.


## 1. Dữ liệu

In [2]:
train = pd.read_csv('data/train.csv'); test = pd.read_csv('data/test.csv')
print('Train:', train.shape, '| Test:', test.shape)
feat = train.drop(columns=['Id', 'SalePrice'])
n_num = feat.select_dtypes('number').shape[1]; n_cat = feat.shape[1] - n_num
print(f'Số biến dự báo: {feat.shape[1]} (số học {n_num}, định tính {n_cat})')
miss = (feat.isna().mean() * 100).sort_values(ascending=False)
print('Top biến khuyết nhiều nhất (%):'); print(miss.head(6).round(1))
RES.update(n_train=len(train), n_test=len(test), n_feat=feat.shape[1], n_num=n_num, n_cat=n_cat,
           miss_top={k: round(v, 1) for k, v in miss.head(6).items()})

Train: (1460, 81) | Test: (1459, 80)
Số biến dự báo: 79 (số học 36, định tính 43)
Top biến khuyết nhiều nhất (%):
PoolQC         99.5
MiscFeature    96.3
Alley          93.8
Fence          80.8
MasVnrType     59.7
FireplaceQu    47.3
dtype: float64


## 2. Biến mục tiêu và điểm ngoại lai

In [3]:
sk0 = train.SalePrice.skew(); sk1 = np.log1p(train.SalePrice).skew()
print(f'Skewness SalePrice: {sk0:.2f}  →  log1p: {sk1:.2f}')
out_idx = train[(train.GrLivArea > 4000) & (train.SalePrice < 300000)].index
print(train.loc[out_idx, ['Id', 'GrLivArea', 'SalePrice', 'SaleCondition']])
RES.update(skew_raw=round(sk0, 2), skew_log=round(sk1, 2), outlier_ids=train.loc[out_idx, 'Id'].tolist(),
           outlier_cond=train.loc[out_idx, 'SaleCondition'].tolist())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(train.SalePrice, bins=50, color=BLUE, edgecolor='white'); ax[0].set_title(f'SalePrice gốc (skewness = {sk0:.2f})'); ax[0].set_xlabel('SalePrice ($)')
ax[1].hist(np.log1p(train.SalePrice), bins=50, color=GREEN, edgecolor='white'); ax[1].set_title(f'log1p(SalePrice) (skewness = {sk1:.2f})'); ax[1].set_xlabel('log(1 + SalePrice)')
for a in ax: a.set_ylabel('Số căn nhà')
plt.tight_layout(); plt.savefig('figs/fig2_target_distribution.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.scatter(train.GrLivArea, train.SalePrice, s=10, alpha=.45, color=BLUE, label='Các căn nhà còn lại')
o = train.loc[out_idx]; ax.scatter(o.GrLivArea, o.SalePrice, s=60, color='crimson', label='2 điểm ngoại lai (Id %s)' % ', '.join(map(str, o.Id)))
ax.axvline(4000, ls='--', color=GREY); ax.axhline(300000, ls=':', color=GREY)
ax.set_xlabel('GrLivArea (sq ft)'); ax.set_ylabel('SalePrice ($)'); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig('figs/fig3_outliers.png', dpi=200); plt.show()

Skewness SalePrice: 1.88  →  log1p: 0.12
        Id  GrLivArea  SalePrice SaleCondition
523    524       4676     184750       Partial
1298  1299       5642     160000       Partial


## 3. Tiền xử lý
- Loại 2 outlier (GrLivArea > 4000 và SalePrice < 300.000$; cả hai đều là giao dịch `Partial`).
- Biến phân loại: `NA` → `'None'`, mã hóa nhãn (LabelEncoder tương đương; tập nhãn lấy từ train + test chỉ để đồng bộ mã, không dùng target).
- Biến số: cột diện tích/đếm của tiện ích không tồn tại → 0; `LotFrontage` → median theo Neighborhood (tính trên train); các cột còn lại → median của train.
- Chuẩn hóa `StandardScaler` được đặt trong Pipeline (fit riêng cho từng fold).

In [4]:
ZERO = ['MasVnrArea','GarageYrBlt','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','GarageCars','GarageArea']

def load(drop_outliers=True):
    tr = pd.read_csv('data/train.csv'); te = pd.read_csv('data/test.csv')
    if drop_outliers:
        tr = tr.drop(tr[(tr.GrLivArea > 4000) & (tr.SalePrice < 300000)].index).reset_index(drop=True)
    y = tr.SalePrice.values; ids = te.Id.values
    X = tr.drop(['Id', 'SalePrice'], axis=1); T = te.drop(['Id'], axis=1)
    cat = X.select_dtypes(exclude='number').columns.tolist(); num = X.select_dtypes('number').columns.tolist()
    med = X.groupby('Neighborhood').LotFrontage.median(); g = X.LotFrontage.median()
    for D in (X, T): D['LotFrontage'] = D.LotFrontage.fillna(D.Neighborhood.map(med)).fillna(g)
    for c in num:
        for D in (X, T): D[c] = D[c].fillna(0 if c in ZERO else X[c].median())
    for c in cat:
        X[c] = X[c].fillna('None').astype(str); T[c] = T[c].fillna('None').astype(str)
        mp = {k: i for i, k in enumerate(sorted(set(X[c]) | set(T[c])))}
        X[c] = X[c].map(mp); T[c] = T[c].map(mp)
    return X, y, T, ids

X, y, T, ids = load(True); ylog = np.log1p(y)
print('X:', X.shape, '| T:', T.shape, '| thiếu sau xử lý:', int(X.isna().sum().sum()), int(T.isna().sum().sum()))
RES['n_train_clean'] = len(X)

X: (1458, 79) | T: (1459, 79) | thiếu sau xử lý: 0 0


## 4. Tương quan

In [5]:
raw = pd.read_csv('data/train.csv').drop(index=out_idx).reset_index(drop=True)
num_raw = raw.drop(columns=['Id']).select_dtypes('number')      # chỉ biến số thật (không dùng nhãn mã hóa), tương quan theo cặp
corr = num_raw.corr()
top = corr['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False).head(11).index.tolist()
r_top = {c: round(corr.loc[c, 'SalePrice'], 2) for c in top}
print('Tương quan Pearson với SalePrice:', r_top)
pairs = [('GarageCars','GarageArea'),('TotalBsmtSF','1stFlrSF'),('YearBuilt','GarageYrBlt')]
r_pairs = {f'{a}~{b}': round(corr.loc[a, b], 2) for a, b in pairs}; print('Cặp đa cộng tuyến:', r_pairs)
RES.update(r_top=r_top, r_pairs=r_pairs)
cols = ['SalePrice'] + top[:10]
fig, ax = plt.subplots(figsize=(7.5, 6.2))
sns.heatmap(corr.loc[cols, cols], annot=True, fmt='.2f', cmap='Blues', annot_kws={'size': 8}, cbar_kws={'shrink': .8}, ax=ax)
plt.tight_layout(); plt.savefig('figs/fig4_correlation.png', dpi=200); plt.show()

Tương quan Pearson với SalePrice: {'OverallQual': np.float64(0.8), 'GrLivArea': np.float64(0.73), 'TotalBsmtSF': np.float64(0.65), 'GarageCars': np.float64(0.64), '1stFlrSF': np.float64(0.63), 'GarageArea': np.float64(0.63), 'FullBath': np.float64(0.56), 'TotRmsAbvGrd': np.float64(0.54), 'YearBuilt': np.float64(0.52), 'YearRemodAdd': np.float64(0.51), 'GarageYrBlt': np.float64(0.49)}
Cặp đa cộng tuyến: {'GarageCars~GarageArea': np.float64(0.89), 'TotalBsmtSF~1stFlrSF': np.float64(0.8), 'YearBuilt~GarageYrBlt': np.float64(0.83)}
